# ESP32 Pico Recorder：燒錄與設備編號

音訊錄音韌體來源是專案根目錄的 `src/main.cpp`；板子設定在 `platformio.ini`；音訊腳位與版本在 `include/audio_config.h`。所有板子燒錄**同一份**韌體。SD 卡不參與燒錄、UID 檢查或 Provisioning。

設備編號 `PR-xxx` 不在編譯時寫死，而是在韌體燒錄後由 USB Provisioning 寫入 NVS。

## 正確流程

**全新板／沒有目前共用韌體：** 選 port → 燒錄音訊韌體 → 唯讀檢查 UID → 寫入 NVS 與 Registry。

**已燒錄共用韌體的板子：** 選 port → 跳過燒錄 → 唯讀檢查 UID → 僅對 `PR-UNASSIGNED` 新 UID 執行 Provisioning。

唯讀檢查使用韌體提供的 `IDENTITY` 指令，因此不能在未燒錄此韌體的全新板上先執行。

## 1. 設定

一片 ESP32-S3 Pico 可能同時出現原生 USB JTAG 與 CH340 UART 兩個 port。本韌體使用原生 USB JTAG（VID:PID `303A:1001`）；CH340（如 `1A86:55D3`）不會被選用。

In [168]:
from pathlib import Path
import csv
import json
import re
import shlex
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'platformio.ini').exists():
    PROJECT_ROOT = Path('/Users/fugu/MEGA/esp32wifi_test_tcp/esp32s3_pico_bluetooth_doc')
if not (PROJECT_ROOT / 'platformio.ini').exists():
    raise RuntimeError('找不到 esp32s3_pico_bluetooth_doc 專案根目錄。')

PLATFORMIO_PYTHON = Path('/Users/fugu/.local/pipx/venvs/platformio/bin/python')
REGISTRY_PATH = PROJECT_ROOT / 'docs/device-registry.csv'
RECORDER_PORT = None  # None：自動選擇唯一的 ESP32-S3 原生 USB port

# 全新板：兩個開關都設為 True，依序燒錄、檢查、Provision。
# 已有共用韌體的板子：保持 CONFIRM_FLASH_AUDIO_FIRMWARE=False。
CONFIRM_FLASH_AUDIO_FIRMWARE = True
CONFIRM_PROVISION = True

assert PLATFORMIO_PYTHON.exists(), f'找不到 PlatformIO Python：{PLATFORMIO_PYTHON}'
assert REGISTRY_PATH.exists(), f'找不到 Registry：{REGISTRY_PATH}'
print(f'專案：{PROJECT_ROOT}')
print('音訊韌體：' + str(PROJECT_ROOT / 'src/main.cpp'))
print(f'Registry：{REGISTRY_PATH}')

專案：/Users/fugu/MEGA/esp32wifi_test_tcp/esp32s3_pico_bluetooth_doc
音訊韌體：/Users/fugu/MEGA/esp32wifi_test_tcp/esp32s3_pico_bluetooth_doc/src/main.cpp
Registry：/Users/fugu/MEGA/esp32wifi_test_tcp/esp32s3_pico_bluetooth_doc/docs/device-registry.csv


## 2. 找出正確的 USB port

只會自動選擇 `USB VID:PID=303A:1001`。若同時插了多片 ESP32-S3，Notebook 會停止並要求手動指定 `RECORDER_PORT`。

In [169]:
def run(command, *, cwd=PROJECT_ROOT, check=True):
    print('$', ' '.join(shlex.quote(str(part)) for part in command))
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    if check and result.returncode:
        raise RuntimeError(f'指令失敗，結束碼：{result.returncode}')
    return result

ports_result = run(['pio', 'device', 'list'])
native_esp32_ports = re.findall(
    r'(?m)^(\/dev\/cu\.\S+)\n-+\nHardware ID: USB VID:PID=303A:1001',
    ports_result.stdout,
)
if RECORDER_PORT is None and len(native_esp32_ports) == 1:
    RECORDER_PORT = native_esp32_ports[0]
    print(f'✅ 已自動選擇 ESP32-S3 原生 USB port：{RECORDER_PORT}')
elif RECORDER_PORT is None:
    raise RuntimeError(f'找到 {len(native_esp32_ports)} 個 ESP32-S3 原生 USB port；請在設定區手動指定 RECORDER_PORT。')
else:
    print(f'使用手動指定 port：{RECORDER_PORT}')

$ pio device list
/dev/cu.debug-console
---------------------
Hardware ID: n/a
Description: n/a

/dev/cu.Bluetooth-Incoming-Port
-------------------------------
Hardware ID: n/a
Description: n/a

/dev/cu.ROSECeramicsX
---------------------
Hardware ID: n/a
Description: n/a

/dev/cu.usbmodem1201
--------------------
Hardware ID: USB VID:PID=303A:1001 SER=D0:CF:13:41:1A:70 LOCATION=0-1.2
Description: USB JTAG/serial debug unit

/dev/cu.usbmodem5B140807171
---------------------------
Hardware ID: USB VID:PID=1A86:55D3 SER=5B14080717 LOCATION=0-1.4
Description: USB Single Serial
✅ 已自動選擇 ESP32-S3 原生 USB port：/dev/cu.usbmodem1201


## 3. 燒錄共用音訊韌體（只限全新板）

這一步將 `src/main.cpp` 編譯成 ESP32-S3 韌體並寫入所選板子。一般燒錄不會清除 NVS；已配號板若需要更新同一份韌體，也可燒錄，但日常 Provisioning 不需要重燒。

In [170]:
flash_command = ['pio', 'run', '--target', 'upload', '--upload-port', RECORDER_PORT]
if CONFIRM_FLASH_AUDIO_FIRMWARE:
    run(flash_command)
else:
    print('跳過燒錄。全新板請將 CONFIRM_FLASH_AUDIO_FIRMWARE 改為 True。')
    print('$', ' '.join(shlex.quote(part) for part in flash_command))

$ pio run --target upload --upload-port /dev/cu.usbmodem1201
Processing esp32-s3-devkitc-1 (platform: espressif32; board: esp32-s3-devkitc-1; framework: arduino)
--------------------------------------------------------------------------------
Verbose mode can be enabled via `-v, --verbose` option
CONFIGURATION: https://docs.platformio.org/page/boards/espressif32/esp32-s3-devkitc-1.html
PLATFORM: Espressif 32 (55.3.38) > Espressif ESP32-S3-DevKitC-1-N8 (8 MB QD, No PSRAM)
HARDWARE: ESP32S3 240MHz, 320KB RAM, 8MB Flash
DEBUG: Current (esp-builtin) On-board (esp-builtin) External (cmsis-dap, esp-bridge, esp-prog, esp-prog-2, iot-bus-jtag, jlink, minimodule, olimex-arm-usb-ocd, olimex-arm-usb-ocd-h, olimex-arm-usb-tiny-h, olimex-jtag-tiny, tumpa)
PACKAGES: 
 - contrib-piohome @ 3.4.4 
 - framework-arduinoespressif32 @ 3.3.8 
 - framework-arduinoespressif32-libs @ 5.5.4+sha.735507283d 
 - tool-esp-rom-elfs @ 2024.10.11 
 - tool-esptoolpy @ 5.2.0 
 - tool-openocd-esp32 @ 2.1200.20260304 (12.

## 4. 唯讀檢查 UID 與 Registry

這一步不會燒錄、不會寫入 NVS、也不會修改 CSV。`new_uid` + `PR-UNASSIGNED` 表示未使用的新板；`registered` 表示已配號。全新板請先完成第 3 步，再執行這一步。

In [171]:
inspect_command = [
    str(PLATFORMIO_PYTHON), 'tools/provision_device.py',
    '--port', RECORDER_PORT, '--registry', str(REGISTRY_PATH), '--inspect',
]
inspection_result = run(inspect_command, check=False)
inspection = None
if inspection_result.returncode == 0:
    inspection = json.loads(inspection_result.stdout.strip())
    if inspection['status'] == 'new_uid' and inspection['device_code'] == 'PR-UNASSIGNED' and not inspection['provision_locked']:
        print(f"✅ 未使用的新板：UID={inspection['hardware_uid']}；將配發 {inspection['suggested_device_code']}")
    elif inspection['status'] == 'registered' and inspection['registry_matches_nvs']:
        print(f"⛔ 已配號：{inspection['registry_device_code']}（UID={inspection['hardware_uid']}）；不要執行第 5 步。")
    else:
        print('⛔ 裝置或 Registry 狀態不一致；請停止並檢查輸出。')
else:
    print('⚠️ 無法讀取 IDENTITY：請確認第 3 步已完成、板子已重新開機，且選到原生 USB port。')

$ /Users/fugu/.local/pipx/venvs/platformio/bin/python tools/provision_device.py --port /dev/cu.usbmodem1201 --registry /Users/fugu/MEGA/esp32wifi_test_tcp/esp32s3_pico_bluetooth_doc/docs/device-registry.csv --inspect
{"status": "new_uid", "hardware_uid": "D0CF13411A70", "device_code": "PR-UNASSIGNED", "provision_locked": false, "registry_device_code": "", "registry_status": "", "registry_matches_nvs": false, "suggested_device_code": "PR-030", "registry": "/Users/fugu/MEGA/esp32wifi_test_tcp/esp32s3_pico_bluetooth_doc/docs/device-registry.csv"}
✅ 未使用的新板：UID=D0CF13411A70；將配發 PR-030


## 5. 寫入 NVS 並更新 Registry

僅在第 4 步顯示「未使用的新板」時執行。工具會再讀一次 UID、配發下一個未使用的 `PR-xxx`、寫入並鎖定 NVS，驗證後才更新 Registry。

In [172]:
provision_command = [
    str(PLATFORMIO_PYTHON), 'tools/provision_device.py',
    '--port', RECORDER_PORT, '--registry', str(REGISTRY_PATH), '--auto',
]
is_verified_new_board = bool(inspection and inspection['status'] == 'new_uid' and inspection['device_code'] == 'PR-UNASSIGNED' and not inspection['provision_locked'])
if CONFIRM_PROVISION and is_verified_new_board:
    run(provision_command)
elif CONFIRM_PROVISION:
    print('⛔ 未通過「未使用新板」的唯讀檢查；不寫入 NVS 或 Registry。')
else:
    print('尚未 Provision。確認第 4 步顯示「未使用的新板」後，將 CONFIRM_PROVISION 改為 True。')
    print('$', ' '.join(shlex.quote(part) for part in provision_command))

$ /Users/fugu/.local/pipx/venvs/platformio/bin/python tools/provision_device.py --port /dev/cu.usbmodem1201 --registry /Users/fugu/MEGA/esp32wifi_test_tcp/esp32s3_pico_bluetooth_doc/docs/device-registry.csv --auto
{"status": "provisioned", "device_code": "PR-030", "hardware_uid": "D0CF13411A70", "registry": "/Users/fugu/MEGA/esp32wifi_test_tcp/esp32s3_pico_bluetooth_doc/docs/device-registry.csv"}


## 6. 查看 Registry

`provisioned` 表示已寫入 NVS 並完成登錄；`unassigned` 是可配發的號碼，並不代表已知新板。

In [173]:
with REGISTRY_PATH.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
active_rows = [row for row in rows if row['status'].strip().lower() != 'unassigned']
for row in active_rows:
    print(json.dumps({key: row[key] for key in ('device_code', 'hardware_uid', 'status', 'firmware_version', 'notes')}, ensure_ascii=False))
print(f'已配發／保留：{len(active_rows)} 台；可配發：{len(rows) - len(active_rows)} 台')

{"device_code": "PR-001", "hardware_uid": "D0CF13411A50", "status": "provisioned", "firmware_version": "5.4.2", "notes": "Legacy migration: assigned before USB Provisioning policy"}
{"device_code": "PR-002", "hardware_uid": "D0CF134113F8", "status": "provisioned", "firmware_version": "5.4.2", "notes": ""}
{"device_code": "PR-003", "hardware_uid": "D0CF13411394", "status": "provisioned", "firmware_version": "5.4.2", "notes": ""}
{"device_code": "PR-004", "hardware_uid": "D0CF13411A64", "status": "provisioned", "firmware_version": "5.4.2", "notes": ""}
{"device_code": "PR-005", "hardware_uid": "D0CF134115A4", "status": "provisioned", "firmware_version": "5.4.2", "notes": ""}
{"device_code": "PR-006", "hardware_uid": "D0CF134113C0", "status": "provisioned", "firmware_version": "5.4.2", "notes": ""}
{"device_code": "PR-007", "hardware_uid": "D0CF134115B8", "status": "provisioned", "firmware_version": "5.4.2", "notes": ""}
{"device_code": "PR-008", "hardware_uid": "D0CF13411A8C", "status": 